# MadMario — GPU Training on Google Colab

**Before running:** `Runtime → Change runtime type → T4 GPU`

This notebook clones the MadMario repo, installs dependencies, and trains on GPU.

## 1. Check GPU

In [ ]:
import subprocess, sys
print(f"Python: {sys.version}")
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                    '--format=csv,noheader'], capture_output=True, text=True)
print("GPU:", r.stdout.strip() or 'NOT FOUND — check Runtime type')
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("No GPU detected — go to Runtime → Change runtime type → T4 GPU")

## 2. Clone Repo & Install

In [ ]:
import os

REPO_URL = 'https://github.com/ohsono/MadMario.git'  # public HTTPS

# If your repo is private, use a personal access token:
# GITHUB_TOKEN = 'ghp_xxxxxxxxxxxx'   # paste your token here
# REPO_URL = f'https://{GITHUB_TOKEN}@github.com/ohsono/MadMario.git'

if os.path.exists('MadMario'):
    print('Repo already cloned — pulling latest')
    !cd MadMario && git pull
else:
    !git clone {REPO_URL}

%cd MadMario
!git log --oneline -3

In [ ]:
%%bash
set -e

# Virtual framebuffer (needed for headless video rendering)
apt-get install -qq xvfb > /dev/null
pip install -q pyvirtualdisplay

# gym-super-mario-bros: >=9.0 requires Python 3.13+; fallback to 7.4.0+shimmy
python_minor=$(python3 -c 'import sys; print(sys.version_info.minor)')
if [ "$python_minor" -ge 13 ]; then
    echo "Python 3.$python_minor: installing gym-super-mario-bros>=9.0"
    pip install -q -e '.[dev]'
else
    echo "Python 3.$python_minor: installing with shimmy compatibility layer"
    pip install -q gymnasium 'gym>=0.26' 'gym-super-mario-bros==7.4.0' \
        'nes-py==8.2.1' 'shimmy>=1.0' opencv-python-headless \
        numpy matplotlib typer torch
    # Install package in editable mode without the pinned mario deps
    pip install -q -e . --no-deps
fi

echo "Done."

In [ ]:
# Patch environment.py for shimmy compat on Python < 3.13
# Skip this cell if you're on Python 3.13+
import sys
if sys.version_info.minor < 13:
    env_src = open('madmario/environment.py').read()
    if '_make_base_env' not in env_src:
        patch = '''
def _make_base_env(env_name, render_mode):
    import gym_super_mario_bros, sys
    if sys.version_info.minor >= 13:
        return gym_super_mario_bros.make(env_name, render_mode=render_mode)
    from shimmy.gym_utils import GymV21CompatibilityV0
    base = gym_super_mario_bros.make(env_name)
    return GymV21CompatibilityV0(base, render_mode=render_mode)
'''
        # Replace the make call inside make_env
        env_src = env_src.replace(
            'env = gym_super_mario_bros.make(config.env_name, render_mode=render_mode)',
            'env = _make_base_env(config.env_name, render_mode)'
        )
        # Add helper before make_env
        env_src = env_src.replace('def make_env(', patch + '\ndef make_env(')
        open('madmario/environment.py', 'w').write(env_src)
        print('Patched environment.py for Python 3.x shimmy compat')
    else:
        print('environment.py already patched')
else:
    print('Python 3.13+ — no patch needed')

In [ ]:
from pyvirtualdisplay import Display
_display = Display(visible=False, size=(640, 480))
_display.start()
print('Virtual display started')

## 3. Google Drive — Checkpoint Persistence

In [ ]:
USE_DRIVE = True   # set False to skip

DRIVE_DIR = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/MadMario'
    import os; os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f'Drive mounted → {DRIVE_DIR}')
else:
    print('Skipping Drive — files lost on session end')

In [ ]:
import os, shutil
from pathlib import Path

SAVE_DIR = Path('runs/colab')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = None

# --- Option A: Upload from your machine ---
UPLOAD_FROM_LOCAL = False
if UPLOAD_FROM_LOCAL:
    from google.colab import files
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    dest = SAVE_DIR / fname
    shutil.move(fname, dest)
    CHECKPOINT_PATH = dest
    print(f'Uploaded: {CHECKPOINT_PATH}')

# --- Option B: Auto-load from Drive ---
if CHECKPOINT_PATH is None and DRIVE_DIR:
    drive_ckpt = Path(DRIVE_DIR) / 'best.chkpt'
    if drive_ckpt.exists():
        # Copy to local for faster I/O during training
        local = SAVE_DIR / 'best.chkpt'
        shutil.copy(drive_ckpt, local)
        CHECKPOINT_PATH = local
        print(f'Loaded from Drive → {CHECKPOINT_PATH}')

if CHECKPOINT_PATH is None:
    print('No checkpoint — training from scratch')
else:
    print(f'Resuming from: {CHECKPOINT_PATH}')

## 4. Train

In [ ]:
# ── Settings ─────────────────────────────────────────────────
EPISODES     = 10_000
PRINT_EVERY  = 100
SEED         = 42
# ─────────────────────────────────────────────────────────────

import random, time
import numpy as np
import torch
from madmario.config import Config, EnvConfig, AgentConfig, TrainConfig
from madmario.environment import make_env
from madmario.agent import Mario

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

seed_everything(SEED)

cfg = Config(
    env=EnvConfig(world=1, stage=1),
    agent=AgentConfig(
        burnin=10_000,       # shorter for Colab (full 100k wastes ~10 min)
        memory_size=50_000,  # fits in 12 GB Colab RAM; use 100k on Pro
        save_every=200_000,
    ),
)

env = make_env(cfg.env, seed=SEED)
mario = Mario(
    state_dim=cfg.state_dim,
    action_dim=env.action_space.n,
    save_dir=SAVE_DIR,
    config=cfg.agent,
    checkpoint=CHECKPOINT_PATH,
)
print(f'\nReady — {EPISODES} episodes on {mario.device}')

In [ ]:
history = []
best_mean = -float('inf')
best_path = SAVE_DIR / 'best.chkpt'
t0 = time.time()

for episode in range(1, EPISODES + 1):
    obs, _ = env.reset()
    total_reward, flag_get = 0.0, False
    while True:
        action = mario.act(obs)
        next_obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        mario.cache(obs, next_obs, action, reward, done)
        mario.learn()
        obs = next_obs
        total_reward += reward
        if done or info.get('flag_get'):
            flag_get = bool(info.get('flag_get'))
            break

    history.append({'episode': episode, 'reward': total_reward, 'flag_get': flag_get})

    if episode >= 20:
        mean20 = float(np.mean([h['reward'] for h in history[-20:]]))
        if mean20 > best_mean:
            best_mean = mean20
            mario.save(best_path)
            # Mirror to Drive immediately so we don't lose it on disconnect
            if DRIVE_DIR:
                shutil.copy(best_path, Path(DRIVE_DIR) / 'best.chkpt')

    if episode % PRINT_EVERY == 0:
        recent  = [h['reward'] for h in history[-20:]]
        flags   = sum(h['flag_get'] for h in history[-100:])
        elapsed = time.time() - t0
        print(f"ep {episode:>6}/{EPISODES} | "
              f"20-ep mean {np.mean(recent):>7.1f} | "
              f"best {best_mean:>7.1f} | "
              f"ε {mario.exploration_rate:.4f} | "
              f"flags/100 {flags} | "
              f"{episode/(elapsed/3600):.0f} ep/hr")

mario.save()
env.close()

total_flags = sum(h['flag_get'] for h in history)
print(f"\n{'='*60}")
print(f"Done: {EPISODES} eps in {(time.time()-t0)/3600:.2f}h")
print(f"Best 20-ep mean : {best_mean:.1f}")
print(f"Flag completions: {total_flags}/{EPISODES} ({total_flags/EPISODES:.1%})")

## 5. Record Gameplay Videos

In [ ]:
import cv2
from IPython.display import Video, display
from pathlib import Path

VIDEO_DIR  = Path('videos'); VIDEO_DIR.mkdir(exist_ok=True)
N_EPISODES = 5
EPS        = 0.02

rec_env = make_env(cfg.env, seed=0, render_mode='rgb_array')
rec_mario = Mario(
    state_dim=cfg.state_dim,
    action_dim=rec_env.action_space.n,
    save_dir=SAVE_DIR,
    config=cfg.agent,
    checkpoint=best_path,
)
rec_mario.exploration_rate = EPS
rec_mario.cfg.exploration_rate_min = EPS

recorded = []
for ep in range(N_EPISODES):
    obs, _ = rec_env.reset()
    frames = [rec_env.render()]
    total, done, info = 0.0, False, {}
    while not done:
        action = rec_mario.act(obs)
        obs, rew, term, trunc, info = rec_env.step(action)
        frames.append(rec_env.render())
        total += rew
        done = term or trunc or bool(info.get('flag_get'))

    flag = bool(info.get('flag_get'))
    name = f"colab_ep{ep}_r{total:.0f}{'_FLAG' if flag else ''}.mp4"
    path = VIDEO_DIR / name
    h, w = frames[0].shape[:2]
    writer = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*'mp4v'), 15, (w, h))
    for f in frames:
        writer.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
    writer.release()
    print(f"ep {ep+1}/{N_EPISODES}: r={total:.0f} flag={flag} ({len(frames)} frames) → {path}")
    recorded.append(str(path))

rec_env.close()

best_vid = max(recorded, key=lambda p: float(
    p.split('_r')[1].replace('_FLAG','').replace('.mp4','')))
print(f'\nBest: {best_vid}')
display(Video(best_vid, embed=True, width=512))

## 6. Save to Drive & Download

In [ ]:
import shutil, os
from google.colab import files

if DRIVE_DIR and os.path.exists('/content/drive'):
    shutil.copy(best_path, Path(DRIVE_DIR) / 'best.chkpt')
    vid_dest = Path(DRIVE_DIR) / 'videos'
    vid_dest.mkdir(exist_ok=True)
    for v in recorded:
        shutil.copy(v, vid_dest)
    print(f'Saved to Drive: {DRIVE_DIR}')

# Download to your browser
print('Downloading checkpoint...')
files.download(str(best_path))
print(f'Downloading best video: {best_vid}')
files.download(best_vid)

## Notes

**If your repo is private**, replace the `REPO_URL` in cell 2 with:
```python
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/ohsono/MadMario.git'
```
Generate a token at: GitHub → Settings → Developer settings → Personal access tokens

**Session disconnect:** Drive checkpoint is mirrored on every new best — you won't lose progress.

**Colab free tier limits:** ~12h runtime, ~90min idle timeout. T4 GPU gives ~10-15× speedup over local CPU.

**Memory:** `memory_size=50_000` uses ~2.8 GB RAM. Increase to `100_000` on Colab Pro (High-RAM).